In [11]:
import os
import json
import openai
import re
import random
import importlib
import matplotlib.pyplot as plt
from openai import OpenAI

In [12]:
# Load the API key into the client for OpenAI API access.
def load_api_key(file_path='api_key.txt'):
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('api_key='):
                return line.strip().split('=', 1)[1]
    return None

api_key = load_api_key()

if api_key is None:
    print("Error: API key not found.")
    exit()

client = OpenAI(api_key=api_key,)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Say this is a test",
        }
    ],
    model="gpt-4o-mini",
)

In [13]:
# Load prompts from the file
def load_prompts_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()  # Read the entire file content

    # Split the content into system and user sections using markers
    system_prompt = content.split('--USER--')[0].replace('--SYSTEM--', '').strip()
    user_prompt = content.split('--USER--')[1].strip()
    
    return system_prompt, user_prompt

In [14]:
# Function to generate a schedule
def generate_weekly_schedule(personality_description, room_list, prompt_path, client):
    
    system_prompt, user_prompt = load_prompts_from_file(prompt_path)
    
    days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    
    weekly_schedule = ""
    room_list_str = ', '.join(room_list)
    
    prompt_personality_part = (
        f'The personality description is as follows: "{personality_description}"'
    )

    prompt_environment_part = (
        f"The schedule should cover the entire day, from wake-up to sleep, using the 24-hour time format. "
        f"For each time slot, vary the start and end times, avoiding times divisible by 5 or 10 minutes. "
        f"Activities happening at home must be in one of the following rooms: {room_list_str}."
    )
    
    for day in days_of_week:
        prompt_date_part = (
            f"Based on the provided description (job, age, health, and personality), "
            f"generate a daily schedule for this person on {day}."
        )
        
        # Generate the schedule using GPT
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": (
                        prompt_personality_part + " "
                        + prompt_date_part + " "
                        + prompt_environment_part + " "
                        + user_prompt
                    ),
                },
            ],
        )
        
        day_schedule = response.choices[0].message.content
        
        weekly_schedule += f"{day}:\n{day_schedule}\n\n"
    
    return weekly_schedule

In [15]:
# Extract wake-up and sleep times from a day's schedule based on the first and last time slots
def extract_times(day_schedule):
    time_slot_pattern = re.compile(r'\((\d{2}:\d{2}) - (\d{2}:\d{2})\)')
    matches = time_slot_pattern.findall(day_schedule)

    if matches:
        wakeup_time = matches[0][0]
        sleep_time = matches[-1][1]
        return wakeup_time, sleep_time
    else:
        return None, None

# Split a day's schedule into individual activity slots based on line breaks
def split_day_into_slots(day_schedule):
    return day_schedule.split('\n')

In [16]:
# Post-process the weekly schedule by linking each day's sleep time to the next day's wake-up time
def post_process_weekly_schedule(schedule):
    # Split the schedule into days
    day_schedules = schedule.split("\n\n")

    for i in range(len(day_schedules) - 1):
        wakeup_time, sleep_time = extract_times(day_schedules[i])
        next_wakeup_time, _ = extract_times(day_schedules[i + 1])

        if sleep_time and next_wakeup_time:
            day_slots = split_day_into_slots(day_schedules[i])
            last_slot = day_slots.pop()
            new_sleep_entry = f"sleep ({sleep_time} - {next_wakeup_time}) (at home)"
            day_slots.append(new_sleep_entry)
            day_schedules[i] = '\n'.join(day_slots)

    return "\n\n".join(day_schedules)

In [17]:
# Save a weekly schedule to a file under the specified output directory
def save_schedule_to_file(character_name, week_schedule, output_dir):

    os.makedirs(output_dir, exist_ok=True)

    filename = f"{character_name}_weekly_schedule.txt"
    file_path = os.path.join(output_dir, filename)

    week_schedule = [entry for entry in week_schedule if entry.strip()]
    week_schedule = week_schedule[:7]

    days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    with open(file_path, "w", encoding="utf-8") as f:
        for day_index, day_schedule in enumerate(week_schedule):
            if days_of_week[day_index] in day_schedule:
                day_schedule = day_schedule.replace(days_of_week[day_index] + ":", "").strip()

            f.write(f"{days_of_week[day_index]}:\n{day_schedule}\n\n")

    print(f"[AgentSense] Schedule saved to: {file_path.replace(os.sep, '/')}")


In [18]:
# Generate and save a schedule for a single personality file
def generate_schedule_from_personality_file(personality_file_path, output_dir, schedule_prompt_path, client):
    
    room_list = ["bathroom", "bedroom", "kitchen", "livingroom"]

    # Load personality description
    with open(personality_file_path, "r", encoding="utf-8") as f:
        description = f.read().strip()

    filename = os.path.basename(personality_file_path)
    character_name = filename.split("_personality")[0]

    print(f"[AgentSense] Generating schedule for {character_name}...")

    # Generate and post-process schedule
    schedule = generate_weekly_schedule(description, room_list, schedule_prompt_path, client)
    processed_schedule = post_process_weekly_schedule(schedule)

    # Save schedule
    save_schedule_to_file(
        character_name=character_name,
        week_schedule=processed_schedule.split("\n\n"),
        output_dir=output_dir
    )

    print(f"[AgentSense] Schedule generation finished for {character_name}.")


In [19]:
# =========================
# Configuration (User-editable)
# =========================

PERSONALITY_FILE = "data/personality_data/Sarah_personality.txt"
SCHEDULE_OUTPUT_DIR = "data/step_2_schedule_data"

# Prompt file for schedule generation
SCHEDULE_PROMPT_PATH = "prompts/schedule_generation_prompt.txt"

# =========================
# Run
# =========================

generate_schedule_from_personality_file(
    personality_file_path=PERSONALITY_FILE,
    output_dir=SCHEDULE_OUTPUT_DIR,
    schedule_prompt_path=SCHEDULE_PROMPT_PATH,
    client=client
);

[AgentSense] Generating schedule for Sarah...
[AgentSense] Schedule saved to: data/schedule_data/Sarah_weekly_schedule.txt
[AgentSense] Schedule generation finished for Sarah.
